# Person Network Explorer

DS 6105 | Summer 2026 — **Know Their Names Capstone**

Search for a person by name or key fragment, then explore their social/family network out to *n* degrees of separation.

- **Node colors** reflect racial classification in the original records (gold = Black, blue = White, green = Mixed, gray = unknown).
- **Edge colors** reflect the relationship predicate (see legend cell below).
- Hover over any node or edge for details.

In [1]:
import pandas as pd
import networkx as nx
import plotly.graph_objects as go
import ipywidgets as widgets
from IPython.display import display, HTML

In [2]:
PERSON = pd.read_parquet('../data/PERSON.parquet')
RELATION = pd.read_parquet('../data/RELATION.parquet').reset_index()

# One directed edge per pair — keep the highest-count assertion when predicates differ
RELATION_g = (
    RELATION
    .sort_values('n_assertions', ascending=False)
    .drop_duplicates(subset=['subject_key', 'object_key'])
)

G = nx.from_pandas_edgelist(
    RELATION_g,
    source='subject_key',
    target='object_key',
    edge_attr=['predicate', 'n_assertions'],
    create_using=nx.DiGraph(),
)
print(f'{len(PERSON):,} persons  |  {G.number_of_nodes():,} graph nodes  |  {G.number_of_edges():,} directed edges')

76,594 persons  |  33,467 graph nodes  |  60,880 directed edges


In [3]:
PREDICATE_COLORS = {
    'wasEnslavedBy':    '#e74c3c',
    'isParentOf':       '#2980b9',
    'isChildOf':        '#5dade2',
    'isSiblingOf':      '#27ae60',
    'isSpouseOf':       '#e91e63',
    'IsFatherOf':       '#1a5276',
    'IsMotherOf':       '#76448a',
    'isGrandParentOf':  '#148f77',
    'isGrandChildOf':   '#1abc9c',
    'isGrandfatherOf':  '#148f77',
    'isGrandmotherOf':  '#76d7c4',
    'isNiblingOf':      '#e67e22',
    'isPiblingOf':      '#ca6f1e',
    'isCousinOf':       '#f39c12',
    'isChildInLawOf':   '#8e44ad',
    'isParentInLawOf':  '#9b59b6',
    'isSiblingInLawOf': '#6c3483',
}

RACE_COLORS = {'b': '#f1c40f', 'w': '#85c1e9', 'm': '#82e0aa', 'x': '#d7dbdd'}


def parse_key(key):
    # Key format: {birth_year}-{norm_race}-{gender}-{name}  (4 parts)
    parts = key.split('-', 3)
    if len(parts) < 4:
        return dict(birth_year='?', norm_race='x', gender='x', name=key)
    raw  = parts[3].replace('_', ' ').strip()
    name = ' '.join(w.capitalize() for w in raw.split()) if raw not in ('x', '') else '(unnamed)'
    return dict(
        birth_year = parts[0] if parts[0] != 'xxxx' else '?',
        norm_race  = parts[1],
        gender     = parts[2],
        name       = name,
    )


def build_ego_graph(center_key, degrees):
    """Return a plotly Figure for the ego network. Renders natively in Jupyter."""
    ego      = nx.ego_graph(G, center_key, radius=degrees, undirected=True)
    ego_und  = ego.to_undirected()
    dist_map = nx.single_source_shortest_path_length(ego_und, center_key)

    # Spring layout — k controls spacing; seed keeps it stable
    pos = nx.spring_layout(ego_und, seed=42, k=2.5 / max(1, len(ego) ** 0.5))

    # ── Edge traces (one per predicate type so colours are consistent) ──
    from collections import defaultdict
    edge_groups = defaultdict(list)
    for u, v, data in ego.edges(data=True):
        edge_groups[data.get('predicate', 'unknown')].append((u, v))

    traces = []
    for pred, pairs in edge_groups.items():
        color = PREDICATE_COLORS.get(pred, '#888')
        xs, ys = [], []
        for u, v in pairs:
            x0, y0 = pos[u]
            x1, y1 = pos[v]
            xs += [x0, x1, None]
            ys += [y0, y1, None]
        traces.append(go.Scatter(
            x=xs, y=ys, mode='lines',
            line=dict(width=1.5, color=color),
            hoverinfo='skip', showlegend=False,
        ))

    # ── Node trace ───────────────────────────────────────────────────────
    node_x, node_y, colors, sizes, borders, hovers, labels = [], [], [], [], [], [], []
    for node in ego.nodes():
        x, y = pos[node]
        node_x.append(x); node_y.append(y)
        info      = parse_key(node)
        dist      = dist_map.get(node, degrees)
        is_center = node == center_key
        colors.append('#f39c12' if is_center else RACE_COLORS.get(info['norm_race'], '#d7dbdd'))
        sizes.append(22 if is_center else max(8, 18 - dist * 4))
        borders.append('#e67e22' if is_center else '#555')
        labels.append(info['name'])
        p = PERSON.loc[node] if node in PERSON.index else None
        hovers.append(
            f"<b>{info['name']}</b><br>"
            f"<span style='color:#aaa;font-size:10px'>{node}</span><br>"
            f"Birth: {info['birth_year']} &nbsp;|&nbsp; "
            f"Race: {info['norm_race'].upper()} &nbsp;|&nbsp; "
            f"Gender: {info['gender'].upper()}<br>"
            f"Mentions: {int(p.n_mentions) if p is not None else '–'} &nbsp;|&nbsp; "
            f"Relations: {int(p.n_relations) if p is not None else 0}"
        )

    traces.append(go.Scatter(
        x=node_x, y=node_y,
        mode='markers+text',
        hoverinfo='text', hovertext=hovers,
        text=labels, textposition='bottom center',
        textfont=dict(size=8, color='#ecf0f1'),
        marker=dict(color=colors, size=sizes, line=dict(color=borders, width=1.5)),
        showlegend=False,
    ))

    return go.Figure(
        data=traces,
        layout=go.Layout(
            paper_bgcolor='#1c1c2e', plot_bgcolor='#1c1c2e',
            margin=dict(b=10, l=10, r=10, t=10),
            height=580, hovermode='closest', showlegend=False,
            xaxis=dict(showgrid=False, zeroline=False, showticklabels=False, showline=False),
            yaxis=dict(showgrid=False, zeroline=False, showticklabels=False, showline=False),
        ),
    )

In [4]:
edge_legend = [
    ('wasEnslavedBy', '#e74c3c'),
    ('isParentOf / IsFatherOf / IsMotherOf', '#2980b9'),
    ('isChildOf', '#5dade2'),
    ('isSiblingOf', '#27ae60'),
    ('isSpouseOf', '#e91e63'),
    ('isGrandParentOf / isGrandChildOf', '#148f77'),
    ('isNiblingOf / isPiblingOf', '#e67e22'),
    ('isCousinOf', '#f39c12'),
    ('in-law relations', '#8e44ad'),
]
node_legend = [
    ('Center person', '#f39c12'),
    ('Black (B)', '#f1c40f'),
    ('White (W)', '#85c1e9'),
    ('Mixed (M)', '#82e0aa'),
    ('Unknown race', '#d7dbdd'),
]

def swatch(color, circle=False):
    r = '50%' if circle else '2px'
    return f'<span style="display:inline-block;width:13px;height:13px;background:{color};border-radius:{r};margin-right:4px;vertical-align:middle"></span>'

edge_html = ''.join(swatch(c) + f'<span style="margin-right:14px;font-size:12px">{l}</span>' for l, c in edge_legend)
node_html = ''.join(swatch(c, True) + f'<span style="margin-right:14px;font-size:12px">{l}</span>' for l, c in node_legend)

display(HTML(
    '<div style="padding:10px 14px;background:#f8f9fa;border-radius:6px;border:1px solid #dee2e6;line-height:2">'
    f'<b>Edge colors</b><br>{edge_html}<br><b>Node colors</b><br>{node_html}'
    '</div>'
))

In [5]:
all_keys = sorted(G.nodes())

search_w = widgets.Text(
    placeholder='Type name or key fragment (min 2 chars)…',
    description='Search:',
    layout=widgets.Layout(width='460px'),
    style={'description_width': '58px'},
)
person_w = widgets.Select(
    options=[],
    description='Person:',
    rows=7,
    layout=widgets.Layout(width='620px'),
    style={'description_width': '58px'},
)
degree_w = widgets.IntSlider(
    value=2, min=1, max=4, step=1,
    description='Degrees:',
    continuous_update=False,
    layout=widgets.Layout(width='360px'),
    style={'description_width': '62px'},
)
info_w = widgets.HTML(
    '<div style="padding:8px 12px;color:#888;font-style:italic">'
    'Search for a person above to begin.</div>'
)

# FigureWidget lives directly in the VBox — no Output wrapper needed.
graph_w = go.FigureWidget(layout=go.Layout(
    paper_bgcolor='#1c1c2e', plot_bgcolor='#1c1c2e',
    height=580, hovermode='closest', showlegend=False,
    margin=dict(b=10, l=10, r=10, t=10),
    xaxis=dict(showgrid=False, zeroline=False, showticklabels=False, showline=False),
    yaxis=dict(showgrid=False, zeroline=False, showticklabels=False, showline=False),
))


def update_list(change):
    q = change['new'].strip().lower()
    person_w.unobserve(redraw, names='value')
    if len(q) < 2:
        person_w.options = []
        person_w.observe(redraw, names='value')
        return
    matches = [k for k in all_keys if q in k][:80]
    person_w.options = matches
    if matches:
        person_w.value = matches[0]
    person_w.observe(redraw, names='value')
    redraw()


def redraw(_change=None):
    key = person_w.value
    if not key:
        return

    info = parse_key(key)
    if key in PERSON.index:
        p = PERSON.loc[key]
        info_w.value = (
            '<div style="padding:8px 14px;background:#f0f3f4;border-radius:6px;font-size:13px">'
            f'<b style="font-size:15px">{info["name"]}</b> &nbsp;&nbsp;'
            f'Birth: <b>{info["birth_year"]}</b> &nbsp;|&nbsp; '
            f'Race: <b>{info["norm_race"].upper()}</b> &nbsp;|&nbsp; '
            f'Gender: <b>{info["gender"].upper()}</b> &nbsp;|&nbsp; '
            f'Mentions: <b>{int(p.n_mentions)}</b> &nbsp;|&nbsp; '
            f'Relations: <b>{int(p.n_relations)}</b>'
            '</div>'
        )

    # FigureWidget trace update:
    # - data = () clears existing traces (empty is always a valid subset)
    # - add_traces() injects the new ones with their own UIDs
    graph_w.data = ()
    if key in G:
        fig = build_ego_graph(key, degree_w.value)
        graph_w.add_traces(list(fig.data))


search_w.observe(update_list, names='value')
person_w.observe(redraw, names='value')
degree_w.observe(redraw, names='value')

display(widgets.VBox([
    widgets.HBox([search_w, degree_w]),
    person_w,
    info_w,
    graph_w,
]))